This script preprocesses TOF-only sensor data for a Transformer-based model.
1. Locate the repository root dynamically so the code works on different machines.
2. Load normalized train and test datasets from parquet files.
3. Select only TOF sensor columns.
4. Group rows by sequence_id so each gesture sequence becomes one sample.
5. Pad or truncate every sequence to a fixed length (TARGET_LEN = 128)
6. Create:
    - TOF feature arrays
    - padding masks
    - 18-class gesture labels
    - binary BFRB vs non-BFRB labels
    - original effective sequence lengths
7. Save all processed outputs as .npy files for later model training.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()

def find_repo_root(start_path: Path):
    cur = start_path
    for _ in range(10):
        if (cur / ".git").exists() or (cur / "data").exists():
            return cur
        cur = cur.parent
    return start_path

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)

DATA_DIR = REPO_ROOT / "data"
PROC_DIR = DATA_DIR / "processed"

train_path = PROC_DIR / "train_normalized.parquet"
test_path  = PROC_DIR / "test_normalized.parquet"

output_dir = PROC_DIR / "Processed_TOF_Transformer"
output_dir.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Train in :", train_path)
print("Test in  :", test_path)
print("Output   :", output_dir)


# Read normalized data

train_DataFrame = pd.read_parquet(train_path)
test_DataFrame  = pd.read_parquet(test_path)


# TOF columns only

TOF_COLS = [c for c in train_DataFrame.columns if c.startswith("tof_")]
print("Number of TOF cols:", len(TOF_COLS))

if len(TOF_COLS) == 0:
    raise ValueError("No TOF columns found. Please check the column names.")


# Check required columns

required_cols = ["sequence_id", "gesture"]
for col in required_cols:
    if col not in train_DataFrame.columns:
        raise ValueError(f"Missing column in train data: {col}")
    if col not in test_DataFrame.columns:
        raise ValueError(f"Missing column in test data: {col}")


# Label mapping

gesture_classes = sorted(train_DataFrame["gesture"].dropna().unique().tolist())
print("Number of gesture classes in train:", len(gesture_classes))
print("Classes:", gesture_classes)

gesture_to_idx = {g: i for i, g in enumerate(gesture_classes)}

BFRB_Gestures = [
    "Cheek - pinch skin",
    "Forehead - pull hairline",
    "Neck - scratch",
    "Neck - pinch skin",
    "Eyelash - pull hair",
    "Eyebrow - pull hair",
    "Forehead - scratch",
    "Above ear - pull hair"
]


# Sequence ids

train_Id = train_DataFrame["sequence_id"].unique()
test_Id  = test_DataFrame["sequence_id"].unique()

print("Train sequence count:", len(train_Id))
print("Test sequence count :", len(test_Id))


# Check sequence length stats

train_lengths = []
for seq_id in train_Id:
    current_seq_data = train_DataFrame[train_DataFrame["sequence_id"] == seq_id]
    train_lengths.append(len(current_seq_data))

test_lengths = []
for seq_id in test_Id:
    current_seq_data = test_DataFrame[test_DataFrame["sequence_id"] == seq_id]
    test_lengths.append(len(current_seq_data))

train_lengths = np.array(train_lengths)
test_lengths = np.array(test_lengths)

print("Train length stats:")
print(" min =", train_lengths.min())
print(" max =", train_lengths.max())
print(" mean =", train_lengths.mean())

print("Test length stats:")
print(" min =", test_lengths.min())
print(" max =", test_lengths.max())
print(" mean =", test_lengths.mean())


# Set target sequence length


TARGET_LEN = 128
print("TARGET_LEN =", TARGET_LEN)



# pad or truncate one sequence

def process_one_sequence_tof(data_matrix, target_len):
    current_len = len(data_matrix)
    feature_dim = data_matrix.shape[1]

    output = np.zeros((target_len, feature_dim), dtype=np.float32)
    mask = np.zeros(target_len, dtype=np.float32)

    if current_len >= target_len:
        output[:, :] = data_matrix[:target_len, :]
        mask[:] = 1.0
    else:
        output[:current_len, :] = data_matrix
        mask[:current_len] = 1.0

    return output, mask


# Build train arrays

data_train_list = []
mask_train_list = []
label_train_18_list = []
label_train_binary_list = []
seq_train_len_list = []

for seq_id in train_Id:
    current_seq_data = train_DataFrame[train_DataFrame["sequence_id"] == seq_id]

    # keep original row order
    current_seq_data = current_seq_data.sort_index()

    raw_values = current_seq_data[TOF_COLS].to_numpy(dtype=np.float32)

    processed_data, processed_mask = process_one_sequence_tof(
        raw_values,
        target_len=TARGET_LEN
    )

    data_train_list.append(processed_data)
    mask_train_list.append(processed_mask)
    seq_train_len_list.append(min(len(raw_values), TARGET_LEN))

    gesture_name = current_seq_data["gesture"].iloc[0]

    label18 = gesture_to_idx.get(gesture_name, -1)
    label_train_18_list.append(label18)

    label_binary = 1 if gesture_name in BFRB_Gestures else 0
    label_train_binary_list.append(label_binary)

data_train = np.array(data_train_list, dtype=np.float32)
mask_train = np.array(mask_train_list, dtype=np.float32)
label_train_18 = np.array(label_train_18_list, dtype=np.int64)
label_train_binary = np.array(label_train_binary_list, dtype=np.int64)
seq_train_len = np.array(seq_train_len_list, dtype=np.int64)


# Build test arrays

data_test_list = []
mask_test_list = []
label_test_18_list = []
label_test_binary_list = []
seq_test_len_list = []

for seq_id in test_Id:
    current_seq_data = test_DataFrame[test_DataFrame["sequence_id"] == seq_id]

    current_seq_data = current_seq_data.sort_index()

    raw_values = current_seq_data[TOF_COLS].to_numpy(dtype=np.float32)

    processed_data, processed_mask = process_one_sequence_tof(
        raw_values,
        target_len=TARGET_LEN
    )

    data_test_list.append(processed_data)
    mask_test_list.append(processed_mask)
    seq_test_len_list.append(min(len(raw_values), TARGET_LEN))

    gesture_name = current_seq_data["gesture"].iloc[0]

    label18 = gesture_to_idx.get(gesture_name, -1)
    label_test_18_list.append(label18)

    label_binary = 1 if gesture_name in BFRB_Gestures else 0
    label_test_binary_list.append(label_binary)

data_test = np.array(data_test_list, dtype=np.float32)
mask_test = np.array(mask_test_list, dtype=np.float32)
label_test_18 = np.array(label_test_18_list, dtype=np.int64)
label_test_binary = np.array(label_test_binary_list, dtype=np.int64)
seq_test_len = np.array(seq_test_len_list, dtype=np.int64)


# Print output shapes

print("data_train shape        :", data_train.shape)
print("mask_train shape        :", mask_train.shape)
print("label_train_18 shape    :", label_train_18.shape)
print("label_train_binary shape:", label_train_binary.shape)

print("data_test shape         :", data_test.shape)
print("mask_test shape         :", mask_test.shape)
print("label_test_18 shape     :", label_test_18.shape)
print("label_test_binary shape :", label_test_binary.shape)


# Save processed results

np.save(output_dir / "data_train_tof.npy", data_train)
np.save(output_dir / "mask_train_tof.npy", mask_train)  
np.save(output_dir / "label_train_18.npy", label_train_18)
np.save(output_dir / "label_train_binary.npy", label_train_binary)
np.save(output_dir / "seq_train_len.npy", seq_train_len)

np.save(output_dir / "data_test_tof.npy", data_test)
np.save(output_dir / "mask_test_tof.npy", mask_test)
np.save(output_dir / "label_test_18.npy", label_test_18)
np.save(output_dir / "label_test_binary.npy", label_test_binary)
np.save(output_dir / "seq_test_len.npy", seq_test_len)

np.save(output_dir / "gesture_classes.npy", np.array(gesture_classes, dtype=object))
np.save(output_dir / "tof_cols.npy", np.array(TOF_COLS, dtype=object))

print("OVER!")

Repo root: E:\A1 final
Train in : E:\A1 final\data\processed\train_normalized.parquet
Test in  : E:\A1 final\data\processed\test_normalized.parquet
Output   : E:\A1 final\data\processed\Processed_TOF_Transformer
Number of TOF cols: 320
Number of gesture classes in train: 18
Classes: ['Above ear - pull hair', 'Cheek - pinch skin', 'Drink from bottle/cup', 'Eyebrow - pull hair', 'Eyelash - pull hair', 'Feel around in tray and pull out an object', 'Forehead - pull hairline', 'Forehead - scratch', 'Glasses on/off', 'Neck - pinch skin', 'Neck - scratch', 'Pinch knee/leg skin', 'Pull air toward your face', 'Scratch knee/leg skin', 'Text on phone', 'Wave hello', 'Write name in air', 'Write name on leg']
Train sequence count: 5959
Test sequence count : 1629
Train length stats:
 min = 35
 max = 700
 mean = 70.34989092129553
Test length stats:
 min = 38
 max = 671
 mean = 71.76181706568447
TARGET_LEN = 128
data_train shape        : (5959, 128, 320)
mask_train shape        : (5959, 128)
label_tra